# Phase 2 - Data Cleaning

*Objective*: Produce two clean, model ready DataFrames from the raw csv. The two DataFrames are:
df_train which will be the data from 2015/16 to 2019/20 for model training and df_holdout which will be the data from 2020/21 to 2021/22 i.e. the out of time test set

In [1]:
import pandas as pd
import numpy as np
import json
import os

for pkg in ['pandas', 'numpy']:
    import importlib
    mod = importlib.import_module(pkg)
    print(f"{pkg}: {mod.__version__}")

pandas: 2.2.3
numpy: 2.1.3


In [4]:
RAW_CSV = '../data/raw/L4_Tidy_2024_ALL_LA.csv'
AUDIT_JSON = 'models/audit_decisions.json'
OUT_DIR = '../data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

with open(AUDIT_JSON) as f:
    cfg = json.load(f)

In [5]:
SENTINELS = list(cfg['sentinels'].keys())
NUM_COLS = cfg['num_cols']
COHORT_BREAK = [int(y) for y in cfg['cohort_break_years']]
MIN_COHORT = cfg['min_cohort_size']
STRATE = ['region_name', 'breakdown']

We will upload the raw csv and apply Phase 1 filters

In [6]:
df = pd.read_csv(RAW_CSV, low_memory=False, encoding='utf-8-sig')
print(f"Raw shape: {df.shape}")

Raw shape: (44342, 30)


From Phase 1, we decided on four filters
<br>The geographic_level will be Local authority
<br>The level_methodology will be Local education authority area
<br>The data_type will be Number of pupils
<br>The institution_group will be State-funded mainstream schools & colleges

In [7]:
df = df[df['geographic_level'] == 'Local authority'].copy()
df = df[df['level_methodology'] == 'Local education authority area'].copy()
df = df[df['data_type']         == 'Number of pupils'].copy()
df = df[df['institution_group'] == 'State-funded mainstream schools & colleges'].copy()

print(f"After filters: {df.shape}")

After filters: (7918, 30)


In [8]:
#WE have to make sure to remove the 'z' (not applicable) values from the numeric cols

z_counts = {c: (df[c]=='z').sum() for c in NUM_COLS if c in df.columns}
assert all(v == 0 for v in z_counts.values()), f"Unexpected 'z' values: {z_counts}"

# Flag suppressed values BEFORE replacement

A suppressed value signals a small cohort, which correlates with the rural LAs, deprived communities, and specialist provision. Capturing this as learned signal is methodologically stronger than silently replacing it.

In [10]:
for col in NUM_COLS:
    if col in df.columns:
        df[f'{col}_suppressed'] = (df[col] == 'c').astype('int8')

suppression_counts = {
    col: int(df[f'{col}_suppressed'].sum())
    for col in NUM_COLS if f'{col}_suppressed' in df.columns
}

print("Suppression counts per column for the filtered dataset:")
for col, n in suppression_counts.items():
    bar = '|' * (n // 5)
    print(f" {col:<22} {n:>4}  {bar}")

Suppression counts per column for the filtered dataset:
 all_cohort                0  
 all_progressed           12  ||
 all_degree                8  |
 all_top3rd               12  ||
 all_appren               42  ||||||||
 all_htech                50  ||||||||||
 acag_cohort              12  ||
 acag_progressed          24  ||||
 tlev_cohort              76  |||||||||||||||
 tlev_progressed         152  ||||||||||||||||||||||||||||||
 otl3_cohort              78  |||||||||||||||
 otl3_progressed         158  |||||||||||||||||||||||||||||||


# Convert sentinel strings to NAN and cast to float

In [11]:
def to_numeric(val: object) -> float:
    """Convert DfE sentinel strings to NaN; all other values to float.
    Sentinels: 'c' (suppressed), 'z' (N/A), 'x' (unavailable), 'low' (<0.5%), '' (empty).
    '0' is a true 0 and is kept as 0.0"""

    s = str(val).strip()
    if s in ('c', 'z','x','low',''):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

for col in NUM_COLS:
    if col in df.columns:
        df[col] = df[col].apply(to_numeric)

In [13]:
#Verifying all num_cols are numeric
float_check = {c: str(df[c].dtype) for c in NUM_COLS if c in df.columns}
non_float = {c: t for c, t in float_check.items() if t != 'float64'}
assert not non_float, f"Non-float columns remain: {non-float}"
print("All numeric columns are float")

All numeric columns are float


In [14]:
#Verify that no sentinel strings remain in the NUM_COLS
for col in NUM_COLS:
    if col in df.columns:
        remaining = df[col].apply(lambda x: isinstance(x, str) and x in SENTINELS).sum()
        assert remaining == 0, f"{remaining} sentinel string remains in {col}"
print("No sentinel string remain in numeric columns")

No sentinel string remain in numeric columns
